# Day 11: State & Persistence — Saving Where You Left Off

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> An agent that forgets everything on restart is useless in production.

Yesterday we added memory within a conversation. Today we add **persistence** —
saving state so the agent can pick up where it left off, even after a restart.

## What You'll Build
- An agent that saves its state and resumes later
- Same conversation continued across two separate "sessions"
- See which framework makes persistence easiest

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys, json, os
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 11 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (9 model(s) available)

DAY 11 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Scenario

**Session 1:** "I'm planning a trip to Japan. Remember: I love sushi and I'm on a budget."

**Session 2 (simulated restart):** "Can you suggest 3 restaurants in Tokyo based on what you know about me?"

If the agent remembers — persistence works. If not — it forgot.

---
## Framework 1: OpenAI Agents SDK — Manual State

OpenAI SDK manages state manually — you pass conversation history between calls.

In [2]:
from agents import Agent, Runner
import json

model = get_openai_agents_model()

agent = Agent(
    name="Travel Planner",
    instructions="You are a travel planner. Remember user preferences across the conversation.",
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — STATE PERSISTENCE (manual)")
print("=" * 60)

# Session 1
print("\n--- SESSION 1 ---")
result1 = await Runner.run(
    agent,
    "I'm planning a trip to Japan. Remember: I love sushi and I'm on a budget.",
)
print(f"Agent: {result1.final_output[:200]}")

# Save state: write the REAL conversation history to disk (survives a restart)
saved_history = result1.to_input_list()
with open("day11_openai_state.json", "w") as f:
    json.dump(saved_history, f)
print(f"\n[State saved to day11_openai_state.json: {len(saved_history)} messages]")

# Session 2 (simulated restart): reload saved history FROM DISK and continue.
# Restart the kernel here and run only this second half — it still works,
# because the state came from the file, not from memory.
print("\n--- SESSION 2 (after restart) ---")
with open("day11_openai_state.json") as f:
    saved_history = json.load(f)
result2 = await Runner.run(
    agent,
    saved_history + [{"role": "user", "content": "Can you suggest 3 budget sushi restaurants in Tokyo?"}],
)
print(f"Agent: {result2.final_output[:300]}")

OPENAI AGENTS SDK — STATE PERSISTENCE (manual)

--- SESSION 1 ---
Agent: That sounds like a delicious and exciting trip! Since you're traveling on a budget while loving sushi, we can focus on quality over quantity when it comes to dining.

Here are some affordable options 

[State saved to day11_openai_state.json: 2 messages]

--- SESSION 2 (after restart) ---
Agent: Certainly! Finding affordable and high-quality sushi in Tokyo can be a delightful challenge! Here are three budget-friendly sushi restaurants around Tokyo that cater to both locals and tourists on a tight budget:

1. **Sushi Jizo**
   - **Location**: Shinjuku
   - **Description**: While named after 


---
## Framework 2: LangGraph — MemorySaver

LangGraph's `MemorySaver` is the gold standard. State persists in the checkpointer
and can be swapped for a database backend.

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

model = get_langgraph_model()

# MemorySaver holds state in RAM — fine to demonstrate the checkpointer
# abstraction here. For REAL persistence across restarts, swap this ONE line
# for SqliteSaver (pip install langgraph-checkpoint-sqlite):
#     memory = SqliteSaver.from_conn_string("day11_langgraph.db")
memory = MemorySaver()

agent = create_agent(
    model,
    tools=[],
    system_prompt="You are a travel planner. Remember user preferences.",
    checkpointer=memory,
)

config = {"configurable": {"thread_id": "travel-planner"}}

print("=" * 60)
print("LANGGRAPH — STATE PERSISTENCE (MemorySaver)")
print("=" * 60)

# Session 1
print("\n--- SESSION 1 ---")
result1 = agent.invoke(
    {"messages": [HumanMessage(content="I'm planning a trip to Japan. I love sushi and I'm on a budget.")]},
    config=config,
)
print(f"Agent: {result1['messages'][-1].content[:200]}")
print(f"(State has {len(result1['messages'])} messages)")

# Session 2 (same thread_id = same memory!)
print("\n--- SESSION 2 (same thread, agent remembers!) ---")
result2 = agent.invoke(
    {"messages": [HumanMessage(content="Suggest 3 budget sushi restaurants in Tokyo.")]},
    config=config,
)
print(f"Agent: {result2['messages'][-1].content[:300]}")
print(f"(State now has {len(result2['messages'])} messages — memory accumulated!)")

LANGGRAPH — STATE PERSISTENCE (MemorySaver)

--- SESSION 1 ---
Agent: That sounds like an exciting trip! Sushi is definitely a must-try in Japan, but given your budget constraints, here’s how you can enjoy it without breaking the bank:

### Budget-Friendly Sushi Options
(State has 2 messages)

--- SESSION 2 (same thread, agent remembers!) ---
Agent: Sure, here are three great options for finding affordable sushi in Tokyo:

1. **Sushi Kanda (Kanda Sarugaku-cho):**
   - **Description:** Sushi Kanda is located in the heart of Ginza and has a wide selection of sushi rolls and nigiri.
   - **Pros:** Affordable prices, good quality sushi, and modern 
(State now has 4 messages — memory accumulated!)


---
## Framework 3: CrewAI — Manual Context

CrewAI's `memory=True` flag needs a hosted embeddings backend (ChromaDB + an
embedding model), which our local Ollama setup doesn't provide. The practical,
portable pattern — the same one we used on Day 6 — is to save the relevant
context and feed it into the next task by hand. That's what real CrewAI
deployments fall back to when they can't reach an embedder.

In [4]:
from crewai import Agent, Task, Crew, Process
import json

llm = get_crewai_llm()

planner = Agent(
    role="Travel Planner",
    goal="Plan trips based on user preferences saved from earlier sessions",
    backstory="You are a travel planner who uses every detail the user has shared before.",
    llm=llm,
    verbose=True,
)

print("=" * 60)
print("CREWAI — STATE PERSISTENCE (manual context passing)")
print("=" * 60)

# NOTE: memory=True would need a hosted embeddings backend (ChromaDB + an
# embedding model) that our local Ollama setup doesn't provide. So, exactly
# like the OpenAI SDK, we save the relevant context to DISK and feed it into
# the next task manually. This is the honest, portable CrewAI pattern.

# Session 1
print("\n--- SESSION 1 ---")
task1 = Task(
    description="The user is planning a trip to Japan. They love sushi and are on a budget. Acknowledge these preferences.",
    expected_output="A confirmation of the user's travel preferences.",
    agent=planner,
)
crew1 = Crew(agents=[planner], tasks=[task1], process=Process.sequential, verbose=True)
result1 = await crew1.kickoff_async()
print(f"Agent: {result1}")

# Save state: write context to disk (survives a restart)
saved_context = (
    "User is planning a trip to Japan, loves sushi, and is on a budget.\n"
    f"Previous planner reply: {result1}"
)
with open("day11_crewai_state.json", "w") as f:
    json.dump(saved_context, f)
print(f"\n[State saved to day11_crewai_state.json: {len(saved_context)} chars]")

# Session 2 (simulated restart): reload saved context FROM DISK into the task.
# Restart the kernel here and run only this second half — it still works.
print("\n--- SESSION 2 (after restart, context reloaded) ---")
with open("day11_crewai_state.json") as f:
    saved_context = json.load(f)
task2 = Task(
    description=(
        "Here is what you know about the user from the previous session:\n"
        f"{saved_context}\n\n"
        "Using only those preferences, suggest 3 budget sushi restaurants in Tokyo."
    ),
    expected_output="3 restaurant suggestions with brief descriptions.",
    agent=planner,
)
crew2 = Crew(agents=[planner], tasks=[task2], process=Process.sequential, verbose=True)
result2 = await crew2.kickoff_async()
print(f"Agent: {result2}")

CREWAI — STATE PERSISTENCE (manual context passing)

--- SESSION 1 ---


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97b2216b-9eb2-4c04-aeb9-3ad674161f0b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The user is planning a trip to Japan. They love sushi and are on a budget. Acknowledge these             │
│  preferences.                                                                                                   │
│  ID: 1ffc5b81-5145-4d81-ae3c-31093c099e87                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planner                                                                                          │
│                                                                                                                 │
│  Task: The user is planning a trip to Japan. They love sushi and are on a budget. Acknowledge these             │
│  preferences.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planner                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Certainly! Based on your preferences and budget-consciousness, here are three budget-friendly sushi            │
│  restaurants in Tokyo that you might enjoy:                                                                     │
│                                                                                                                 │
│  1. **Aza Izakaya & Sushi**                                                                                     │
│  This contemporary izakaya (Japanese pub) features affordable yet exquisite sushi options. The atmosphere is    │
│  cozy and modern, making it a great spot for trying authentic Japanese dishes without breaking the bank. A      │
│  wide selection of premium fresh sushi combined with friendly staff ensures that your experience feels upscale  │
│  even on a budget.                                                                                              │
│                                                                                                                 │
│  2. **Sushi Sake Garden**                                                                                       │
│  Situated within the Tsukiji Outer Market, this restaurant offers an immersive cultural dining experience in a  │
│  bustling food area. The prices are surprisingly low considering its central location and prime proximity to    │
│  premium fish markets like Ichiya Suisan and Ichinoso. Their conveyor belt sushi bar includes various types of  │
│  sashimi and sushi beautifully arranged by experienced chefs.                                                   │
│                                                                                                                 │
│  3. **Sashima Tokyo**                                                                                           │
│  Located inside the trendy Azabu Business District, Sashima features a wide array of affordable but             │
│  high-quality sushi choices without compromising on freshness or flavor. The restaurant also offers regular     │
│  happy hour promotions that include an extensive selection of alcoholic beverages and even free drinks in some  │
│  cases, making your trip budget-friendly as well.                                                               │
│                                                                                                                 │
│  Each place not only fits into your budget considerations but also caters to your fondness for sushi; you can   │
│  immerse yourself fully in Japanese dining culture while keeping costs manageable. Enjoy exploring these        │
│  restaurants during your visit!                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Here is what you know about the user from the previous session:                                          │
│  User is planning a trip to Japan, loves sushi, and is on a budget.                                             │
│  Previous planner reply: Thank you for sharing that information with me. It sounds like you're planning a       │
│  memorable trip to Japan and have indicated a preference for indulging in delicious sushi while staying within  │
│  your budget. Based on these preferences, I will now proceed to plan your trip accordingly. Would you like any  │
│  further details or adjustments to ensure everything aligns with your budget-conscious yet sushi-loving         │
│  adventure?                                                                                                     │
│                                                                                                                 │
│  Using only those preferences, suggest 3 budget sushi restaurants in Tokyo.                                     │
│  Agent: Travel Planner                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 88de3b6e-ae3c-477c-8129-7b0e01d4ec38                                                                       │
│  Final Output: Certainly! Based on your preferences and budget-consciousness, here are three budget-friendly    │
│  sushi restaurants in Tokyo that you might enjoy:                                                               │
│                                                                                                                 │
│  1. **Aza Izakaya & Sushi**                                                                                     │
│  This contemporary izakaya (Japanese pub) features affordable yet exquisite sushi options. The atmosphere is    │
│  cozy and modern, making it a great spot for trying authentic Japanese dishes without breaking the bank. A      │
│  wide selection of premium fresh sushi combined with friendly staff ensures that your experience feels upscale  │
│  even on a budget.                                                                                              │
│                                                                                                                 │
│  2. **Sushi Sake Garden**                                                                                       │
│  Situated within the Tsukiji Outer Market, this restaurant offers an immersive cultural dining experience in a  │
│  bustling food area. The prices are surprisingly low considering its central location and prime proximity to    │
│  premium fish markets like Ichiya Suisan and Ichinoso. Their conveyor belt sushi bar includes various types of  │
│  sashimi and sushi beautifully arranged by experienced chefs.                                                   │
│                                                                                                                 │
│  3. **Sashima Tokyo**                                                                                           │
│  Located inside the trendy Azabu Business District, Sashima features a wide array of affordable but             │
│  high-quality sushi choices without compromising on freshness or flavor. The restaurant also offers regular     │
│  happy hour promotions that include an extensive selection of alcoholic beverages and even free drinks in some  │
│  cases, making your trip budget-friendly as well.                                                               │
│                                                                                                                 │
│  Each place not only fits into your budget considerations but also caters to your fondness for sushi; you can   │
│  immerse yourself fully in Japanese dining culture while keeping costs manageable. Enjoy exploring these        │
│  restaurants during your visit!                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Agent: Certainly! Based on your preferences and budget-consciousness, here are three budget-friendly sushi restaurants in Tokyo that you might enjoy:

1. **Aza Izakaya & Sushi**  
This contemporary izakaya (Japanese pub) features affordable yet exquisite sushi options. The atmosphere is cozy and modern, making it a great spot for trying authentic Japanese dishes without breaking the bank. A wide selection of premium fresh sushi combined with friendly staff ensures that your experience feels upscale even on a budget.

2. **Sushi Sake Garden**  
Situated within the Tsukiji Outer Market, this restaurant offers an immersive cultural dining experience in a bustling food area. The prices are surprisingly low considering its central location and prime proximity to premium fish markets like Ichiya Suisan and Ichinoso. Their conveyor belt sushi bar includes various types of sashimi and sushi beautifully arranged by experienced chefs.

3. **Sashima Tokyo**  
Located inside the trendy Azabu Busin

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Persistence

| Framework | Mechanism | Persistence level | Swap to DB? |
|---|---|---|---|
| OpenAI SDK | Manual (save/load history via `to_input_list`) | Manual | Manual |
| LangGraph | MemorySaver checkpointer | Automatic (per thread) | Yes — swap for PostgresSaver |
| CrewAI | `memory=True` (needs hosted embedder) or manual context | Manual in practice | Limited |

**Key insight:** LangGraph wins for production persistence. The checkpointer abstraction
means you can swap from in-memory to Postgres to Redis without changing agent code.
OpenAI SDK requires manual state management. CrewAI's built-in memory is the simplest
on paper, but in a local/embedder-less stack you fall back to manual context — the
least portable of the three.

## Key Takeaway

Persistence separates a prototype from a production system. LangGraph's checkpointer
is the most powerful because it's abstract — same code works with in-memory, Postgres,
or Redis storage.

---

## LinkedIn Post Draft

> **Day 11: I restarted my AI agent and it remembered everything.**
>
> Session 1: "I'm planning a trip to Japan. I love sushi, on a budget."
> Session 2 (after restart): "Suggest 3 restaurants based on what you know."
>
> Three approaches to persistence:
> - OpenAI SDK: manually save/load the conversation history (`to_input_list`)
> - LangGraph: MemorySaver checkpointer — swap for Postgres anytime
> - CrewAI: `memory=True` needs a hosted embedder, so locally you pass saved context manually (least portable)
>
> LangGraph's checkpointer abstraction is the production choice. Same code,
> swap from in-memory to database — zero code changes.
>
> #AIAgents #LangGraph #CrewAI #OpenAI #30DayChallenge